# Practical 18

**Aim:** Perform data cleaning operations: detect and handle missing values, remove duplicate records, and treat outliers in a dataset.

### 1. Load Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('agaricus-lepiota.csv')
df.replace('?', np.nan, inplace=True)
print('Initial Shape:', df.shape)
df.head(3)

Initial Shape: (8124, 23)


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,e,e,s,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,e,c,s,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,e,c,s,s,w,w,p,w,o,p,n,n,m


### 2. Detect & Handle Missing Values

In [2]:
# Detect missing values
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Handle missing values by imputing Mode
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print('\nRemaining missing values:', df.isnull().sum().sum())

Missing values per column:
stalk-root    2480
dtype: int64

Remaining missing values: 0


### 3. Detect & Remove Duplicate Records

In [3]:
# Detect duplicates
print('Duplicate records count:', df.duplicated().sum())

# Remove duplicates
df.drop_duplicates(inplace=True)
print('Shape after removing duplicates:', df.shape)

Duplicate records count: 0
Shape after removing duplicates: (8124, 23)


### 4. Detect & Treat Outliers

**Note:** The Agaricus/Mushroom dataset contains categorical features only, so a direct IQR calculation on an original column is not appropriate. To keep the same IQR/capping approach as the reference practical, a numeric `rarity_score` is derived from category frequencies. Only this derived score is capped; the original categorical values are not changed.

In [4]:
# Create a numeric rarity score from categorical feature frequencies
feature_cols = df.columns[1:]  # exclude class
df['rarity_score'] = 0.0

for col in feature_cols:
    freq = df[col].value_counts(normalize=True)
    df['rarity_score'] += -np.log10(df[col].map(freq))

# Detect outliers using IQR Method on 'rarity_score'
Q1 = df['rarity_score'].quantile(0.25)
Q3 = df['rarity_score'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['rarity_score'] < lower_bound) | (df['rarity_score'] > upper_bound)]
print(f'IQR: {IQR:.4f}, Lower Bound: {lower_bound:.4f}, Upper Bound: {upper_bound:.4f}')
print('Outliers count in rarity_score:', len(outliers))

# Treat outliers using Capping (Winsorization)
df['rarity_score'] = df['rarity_score'].clip(lower=lower_bound, upper=upper_bound)
print('Outliers treated! Capped Min:', round(df['rarity_score'].min(), 4),
      'Capped Max:', round(df['rarity_score'].max(), 4))

IQR: 2.2482, Lower Bound: 4.5967, Upper Bound: 13.5896
Outliers count in rarity_score: 306
Outliers treated! Capped Min: 6.0128 Capped Max: 13.5896
